In [ ]:
# Importando bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import matplotlib.ticker as ticker
import seaborn as sns
import json
import os
from scipy.stats import skew, kurtosis

In [ ]:
results_dist = ['Resultados_dist_2/result_dist_1_khz.csv', 'Resultados_dist_2/result_dist_10_khz.csv',
                'Resultados_dist_2/result_dist_20_khz.csv', 'Resultados_dist_2/result_dist_pca_1_khz.csv',
                'Resultados_dist_2/result_dist_pca_10_khz.csv', 'Resultados_dist_2/result_dist_pca_20_khz.csv'] #Arquivos .csv contendo os resultados do cross_validate por frequência

In [ ]:
dfs_results = [] #Cria uma lista vazia para armazenar os dataframes
for i in results_dist:
    df = pd.read_csv(i, sep=';', decimal=',') #Lê o arquivo .csv e cria um dataframe com os dados
    dfs_results.append(df) #Armazena o dataframe na lista

In [ ]:
#Função para transformar os dataframes para obter as estatísticas
def create_dfs(df_list):
    df_transformed = [] #Cria uma lista vazia para armazenar os dataframes
    idx_test = ['Fold 1','Fold 2','Fold 3','Fold 4','Fold 5'] #Lista contendo os Folds correspondentes aos dados

    for i in df_list:
        i = i.drop(columns=['fit_time', 'score_time', 'n_samples', 'experiment']) #Exclui colunas que não serão utilizadas
        data = i['test_score'].to_numpy().reshape(-1,5) #Cria um numpy array oredenado com os resultados de acurácia por 5 folds
        df_n = pd.DataFrame(columns=i['model'].unique(), data=data.T, index=idx_test) #Cria dataframe com os resultados de acurácia por modelo, com os Folds como índice
        
        df_transformed.append(df_n) #Armazena o dataframe na lista

    return df_transformed #Retorna lista com os dataframes transformados

In [ ]:
dfs_transformed = create_dfs(dfs_results) #Chama a função e armazena o resultado

In [ ]:
experiments = ['1 kHz', '10 kHz', '20 kHz', '1 kHz + PCA', '10 kHz + PCA', '20 kHz + PCA'] #Lista contendo os experimentos

In [ ]:
dfs_dict = dict(zip(experiments, dfs_transformed)) #Cria dicionário com os experimentos como chaves e os dataframes como dados

In [ ]:
from scipy import stats

def statistic_summary_numpy(df):

    # Filtrando apenas colunas numéricas
    df_array = df.to_numpy().T
    cols = df.columns

    basic_stats = {}

    # Calculando estatísticas básicas
    for i in range(len(df_array)):
        count = len(df_array[i]) #Calcula contagem de dados
        mean = np.mean(df_array[i]) #Calcula média
        std = np.std(df_array[i]) #Calcula desvio padrão
        median = np.median(df_array[i]) #Calcula mediana
        mode = stats.mode(df_array[i])[0][0] #Calcula moda
        min_l = np.min(df_array[i]) #Calcula valor mínimo
        max_l = np.max(df_array[i]) # Calcula valor máximo
        range_l = np.max(df_array[i]) - np.min(df_array[i]) #Calcula alcance dos dados
        quantiles = np.quantile(df_array[i], [0.25, 0.75]) #Calcula os quartis
        q1 = quantiles[0] #Primeiro quartil
        q3 = quantiles[1] #Terceiro quartil
        iqr = q3-q1 #Calcula o IQR
        cv = std/mean #Calcula o coeficiente de variação
        skew_l = skew(df_array[i]) #Calcula o skewness
        kurt_l = kurtosis(df_array[i]) #Calcula kurtosis
        sem = stats.sem(df_array[i]) #Calcula erro padrão da média

        stats_list = [count, mean, std, median, mode, min_l, max_l, range_l, q1, q3, iqr, cv, skew_l, kurt_l, sem] #Lista com as estatísticas
        round_list = [round(x, 4) for x in stats_list] #Arredonda os valores para quatro casas decimais

        basic_stats[cols[i]] = round_list

    statistics_summary_header = ['count', 'mean', 'std', 'median', 'mode', 'min', 'max', 'range', 'Q1', 'Q3', 'IQR', 'cv', 'skewness', 'kurtosis', 'sem'] #Nomes das Colunas

    statistics_summary = pd.DataFrame(basic_stats) #Cria o datafeame

    statistics_summary = statistics_summary.T
    statistics_summary.columns = statistics_summary_header #Define os nomes das colunas

    return statistics_summary #Retorna dataframe com os resultados

In [ ]:
#Função para obter as estatísticas com dicionário
def create_stat(df_dict, keys):
    stats_l = [] #Cria uma lista vazia para armazenar os dataframes
    for i in keys:
        stat = statistic_summary_numpy(df_dict[i]) #Chama a função e cria o dataframe com estatísticas
        stat = stat.reset_index(names='Model') #Transforma o índice em coluna com os modelos
        stat = stat[['Model','mean', 'std', 'range']] #Seleciona apenas as colunas com modelo, média, desvio padrão e range
        stat['experiment'] = i #Cria coluna experiment no dataframe e armazena o valor da chave

        stats_l.append(stat) #Armazena o dataframe na lista

    return stats_l #Retorna a lista com dataframes

In [ ]:
stats_list = create_stat(dfs_dict, experiments) #Chama a função e armazena a lista com os dataframes

C:\Users\Jorge\AppData\Local\Temp\ipykernel_10044\1465803313.py:17: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode = stats.mode(df_array[i])[0][0] #Calcula moda
C:\Users\Jorge\AppData\Local\Temp\ipykernel_10044\1465803313.py:17: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid t

In [ ]:
stats_total = pd.concat(stats_list) #Junta todos os dataframes

In [ ]:
stats_total.to_csv('stats_dist.csv',sep=';', decimal=',', index=False) #Exporta o dataframe como arquivo .csv